In [1]:
import csv

In [2]:
def load_dataset(filename):
  rows = []
  with open(filename , 'r') as file:
    csv_reader = csv.DictReader(file)

    for row in csv_reader:
      rows.append(row)

  return rows

In [3]:
def inspect_transactions(transactions):
  print("Number of transactions is : " , len(transactions))
  print("First transaction is : " , transactions[0])
  print("Last transaction is : " , transactions[-1])

  print("Columns are : " , transactions[0].keys())
  print("First 5 records are : ")
  for transaction in transactions[:5]:
    print(transaction)

  print("Transactions types : ")
  transaction_types = set()
  for transaction in transactions:
    if transaction != 'None':
      transaction_types.add(transaction['type'])


  print(transaction_types)

In [4]:
inspect_transactions(load_dataset('data/Synthetic_Financial_datasets_log.csv'))

Number of transactions is :  6362620
First transaction is :  {'step': '1', 'type': 'PAYMENT', 'amount': '9839.64', 'nameOrig': 'C1231006815', 'oldbalanceOrg': '170136.0', 'newbalanceOrig': '160296.36', 'nameDest': 'M1979787155', 'oldbalanceDest': '0.0', 'newbalanceDest': '0.0', 'isFraud': '0', 'isFlaggedFraud': '0'}
Last transaction is :  {'step': '743', 'type': 'CASH_OUT', 'amount': '850002.52', 'nameOrig': 'C1280323807', 'oldbalanceOrg': '850002.52', 'newbalanceOrig': '0.0', 'nameDest': 'C873221189', 'oldbalanceDest': '6510099.11', 'newbalanceDest': '7360101.63', 'isFraud': '1', 'isFlaggedFraud': '0'}
Columns are :  dict_keys(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'])
First 5 records are : 
{'step': '1', 'type': 'PAYMENT', 'amount': '9839.64', 'nameOrig': 'C1231006815', 'oldbalanceOrg': '170136.0', 'newbalanceOrig': '160296.36', 'nameDest': 'M1979787155', 'oldbalanceDest': '0.

In [5]:
def validate_transactions(transaction):
  try :
    if not transaction["type"]:
      return False , "Missing Transaction Type"
    if not transaction["nameOrig"]:
      return False , "Missing origin account"
    if not transaction["nameDest"]:
      return False , "Missing destination account"

    Accepted_Types = {
            "PAYMENT",
            "TRANSFER",
            "CASH_OUT",
            "DEBIT",
            "CASH_IN",
            "None"
            }
    if transaction["type"] not in Accepted_Types :
      return False , "Invalid Transaction Type"


    step = int(transaction["step"])
    if step < 0:
      return False , "Invalid Step"

    amount = float(transaction["amount"])
    if amount < 0:
      return False , "Invalid Amount"

    oldbalanceOrg = float(transaction["oldbalanceOrg"])
    if oldbalanceOrg < 0:
      return False , "Invalid Old Balance Origin"

    newbalanceOrig = float(transaction["newbalanceOrig"])
    if newbalanceOrig < 0:
      return False , "Invalid New Balance Origin"


    old_balance_dest = float(transaction["oldbalanceDest"])
    new_balance_dest = float(transaction["newbalanceDest"])

    if old_balance_dest < 0 or new_balance_dest < 0:
            return False, "Invalid destination balance"


    return True , "Valid Transaction"
  except (ValueError , TypeError , KeyError) as e:
    return False , str(e)


In [6]:
for transaction in load_dataset('data/Synthetic_Financial_datasets_log.csv'):
  validate_transactions(transaction)

In [7]:
def clean_transaction(transactions):
    cleaned_transactions = []
    invalid_transactions = []

    for transaction in transactions:
        is_valid, reason = validate_transactions(transaction)

        if not is_valid:
            invalid_transactions.append({
                "transaction": transaction,
                "reason": reason
            })
            continue

        cleaned_transaction = {
            "step": int(transaction["step"]),
            "type": transaction["type"].strip(),
            "amount": float(transaction["amount"]),
            "nameOrig": transaction["nameOrig"].strip(),
            "oldbalanceOrg": float(transaction["oldbalanceOrg"]),
            "newbalanceOrig": float(transaction["newbalanceOrig"]),
            "nameDest": transaction["nameDest"].strip(),
            "oldbalanceDest": float(transaction["oldbalanceDest"]),
            "newbalanceDest": float(transaction["newbalanceDest"]),
            "isFraud": int(transaction["isFraud"]),
            "isFlaggedFraud": int(transaction["isFlaggedFraud"])
        }

        cleaned_transactions.append(cleaned_transaction)

    return cleaned_transactions, invalid_transactions

In [8]:
def find_duplicates(transactions):
  unique_transaction = set()
  duplicate_transaction = list()

  for transaction in transactions:
    transaction_key = (
          transaction["step"],
          transaction["type"],
          transaction["amount"],
          transaction["nameOrig"],
          transaction["nameDest"]
    )

    if transaction_key in unique_transaction:
      duplicate_transaction.append(transaction_key)
    else:
      unique_transaction.add(transaction_key)

  return duplicate_transaction


In [9]:
def detect_suspicious_transactions(transactions):

    suspicious_transactions = []

    for transaction in transactions:

        reasons = []

        amount = transaction["amount"]
        transaction_type = transaction["type"]

        old_org = transaction["oldbalanceOrg"]
        new_org = transaction["newbalanceOrig"]

        old_dest = transaction["oldbalanceDest"]
        new_dest = transaction["newbalanceDest"]


        # ---------------------------------------------
        # Rule 1: Very large transaction
        # ---------------------------------------------

        if amount > 100000:
            reasons.append("Very high transaction amount")


        # ---------------------------------------------
        # Rule 2: Origin balance inconsistency
        # ---------------------------------------------

        expected_org_balance = old_org - amount

        if abs(expected_org_balance - new_org) > 0.01:
            reasons.append("Origin balance inconsistency")


        # ---------------------------------------------
        # Rule 3: Transfer/CASH_OUT empties balance
        # ---------------------------------------------

        if transaction_type in {"TRANSFER", "CASH_OUT"}:

            if old_org > 0 and new_org == 0:
                reasons.append("Origin balance emptied")


        # ---------------------------------------------
        # Rule 4: Destination balance inconsistency
        # ---------------------------------------------

        if transaction_type in {"TRANSFER", "CASH_IN"}:

            expected_dest_balance = old_dest + amount

            if abs(expected_dest_balance - new_dest) > 0.01:

                reasons.append(
                    "Destination balance inconsistency"
                )


        # ---------------------------------------------
        # Rule 5: Multiple suspicious conditions
        # ---------------------------------------------

        if len(reasons) >= 2:

            suspicious_transactions.append({
                "transaction": transaction,
                "reasons": reasons
            })


    return suspicious_transactions


# =========================================================
# 7. USER STATISTICS
# =========================================================

def calculate_user_statistics(transactions):

    users = {}

    for transaction in transactions:

        user = transaction["nameOrig"]
        amount = transaction["amount"]

        if user not in users:

            users[user] = {
                "transaction_count": 0,
                "total_amount": 0
            }

        users[user]["transaction_count"] += 1
        users[user]["total_amount"] += amount

    for user in users:
        users[user]["average_amount"] = (
            users[user]["total_amount"] /
            users[user]["transaction_count"]
        )

    return users


In [10]:
def add_suspicious_counts(
    users,
    suspicious_transactions
):

    for user in users:

        users[user]["suspicious_count"] = 0


    for item in suspicious_transactions:

        transaction = item["transaction"]

        user = transaction["nameOrig"]

        if user in users:

            users[user]["suspicious_count"] += 1


    return users


In [11]:
def generate_report(transactions ,  invalid_transactions,
    duplicate_transactions,suspicious_transactions,
    users):
  total_amount = 0
  flagged_count = 0
  fraud_count = 0
  highest_transactions = None

  for transaction in transactions :
    total_amount += float(transaction['amount'])
    if transaction['isFlaggedFraud'] == 1:
      flagged_count += 1

    if transaction['isFraud'] == 1:
      fraud_count += 1

    if highest_transactions is None or (transaction['amount'] is not None and float(highest_transactions['amount']) < float(transaction['amount'])):
      highest_transactions = transaction


  average_amount = total_amount / len(transactions)

  suspicious_percentage = (
        len(suspicious_transactions)
        / len(transactions)
        * 100
        if transactions
        else 0
    )

  print("\n" + "=" * 60)
  print("TRANSACTION RISK ANALYSIS REPORT")
  print("=" * 60)


  print("\nDATA QUALITY")
  print("-" * 40)

  print(
        "Total records:",
        len(transactions) + len(invalid_transactions)
    )

  print(
        "Valid records:",
        len(transactions)
    )

  print(
        "Invalid records:",
        len(invalid_transactions)
    )

  print(
        "Duplicate transactions:",
        len(duplicate_transactions)
    )


  print("\nTRANSACTION SUMMARY")
  print("-" * 40)
  print(
        "Total transaction amount:",
        round(total_amount, 2)
    )

  print(
        "Average transaction amount:",
        round(average_amount, 2)
    )


  if highest_transactions:

    print(
        "Highest transaction:",
        highest_transactions["amount"]
    )

    print(
        "Highest transaction type:",
        highest_transactions["type"]
    )


  print("\nRISK ANALYSIS")
  print("-" * 40)

  print(
        "Suspicious transactions:",
        len(suspicious_transactions)
    )

  print(
        "Suspicious percentage:",
        round(suspicious_percentage, 2),
        "%"
    )


  print("\nDATASET LABEL INFORMATION")
  print("-" * 40)
  print(
        "Actual fraud transactions:",
        fraud_count
   )

  print(
        "Flagged fraud transactions:",
        flagged_count
    )


  print("\nTOP USERS BY TRANSACTION VALUE")
  print("-" * 40)


  sorted_users = sorted(
      users.items(),
      key=lambda x: x[1]["total_amount"],
      reverse=True
  )


  for user, stats in sorted_users[:10]:

    print(
        user,
        "| transactions:",
        stats["transaction_count"],
        "| total:",
        round(stats["total_amount"], 2),
        "| average:",
        round(stats["average_amount"], 2),
        "| suspicious:",
        stats["suspicious_count"]
    )

In [12]:
filename = 'data/Synthetic_Financial_datasets_log.csv'


transactions = load_dataset(filename)

print("DATA INSPECTION")
print("=" * 60)

inspect_transactions(transactions)


cleaned_transactions, invalid_transactions = (
    clean_transaction(transactions)
)


duplicate_transactions = find_duplicates(
    cleaned_transactions
)


suspicious_transactions = (
    detect_suspicious_transactions(
        cleaned_transactions
    )
)


users = calculate_user_statistics(
    cleaned_transactions
)


users = add_suspicious_counts(
    users,
    suspicious_transactions
)


generate_report(
    cleaned_transactions,
    invalid_transactions,
    duplicate_transactions,
    suspicious_transactions,
    users
)

DATA INSPECTION
Number of transactions is :  6362620
First transaction is :  {'step': '1', 'type': 'PAYMENT', 'amount': '9839.64', 'nameOrig': 'C1231006815', 'oldbalanceOrg': '170136.0', 'newbalanceOrig': '160296.36', 'nameDest': 'M1979787155', 'oldbalanceDest': '0.0', 'newbalanceDest': '0.0', 'isFraud': '0', 'isFlaggedFraud': '0'}
Last transaction is :  {'step': '743', 'type': 'CASH_OUT', 'amount': '850002.52', 'nameOrig': 'C1280323807', 'oldbalanceOrg': '850002.52', 'newbalanceOrig': '0.0', 'nameDest': 'C873221189', 'oldbalanceDest': '6510099.11', 'newbalanceDest': '7360101.63', 'isFraud': '1', 'isFlaggedFraud': '0'}
Columns are :  dict_keys(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud'])
First 5 records are : 
{'step': '1', 'type': 'PAYMENT', 'amount': '9839.64', 'nameOrig': 'C1231006815', 'oldbalanceOrg': '170136.0', 'newbalanceOrig': '160296.36', 'nameDest': 'M1979787155', 'oldb